In [ ]:

# SIMULATION: LiNGAM vs Mean-Independence Variant
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from lingam import DirectLiNGAM

# 1. Data-generating proces
def simulate_xy(n=500, beta=1.0, scenario="gaussian", seed=None):
    rng = np.random.default_rng(seed)

    # Cause X
    if scenario == "gaussian":
        x = rng.normal(size=n)
        eps = rng.normal(size=n)

    elif scenario == "non_gaussian":
        # centered exponential -> skewed, clearly non-Gaussian
        x = rng.exponential(size=n) - 1.0
        eps = rng.exponential(size=n) - 1.0

        # standardize a bit for comparability
        x = (x - x.mean()) / x.std()
        eps = (eps - eps.mean()) / eps.std()

    else:
        raise ValueError("scenario must be 'gaussian' or 'non_gaussian'")

    # Effect Y
    y = beta * x + eps
    return x, y
    
# 2. Mean-independence score: E[e | X] approx = 0
def mean_independence_score(x, y, degree=3):
    """
    Lower score = residual is less predictable from x
    = more compatible with E[e|x] = 0.
    """
    # linear regression y on x
    X_lin = x.reshape(-1, 1)
    lin = LinearRegression().fit(X_lin, y)
    resid = y - lin.predict(X_lin)

    # Can x nonlinearly predict the residual?
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly = poly.fit_transform(X_lin)
    aux = LinearRegression().fit(X_poly, resid)

    # Use auxiliary R^2 as dependence score
    score = aux.score(X_poly, resid)
    return score


def direction_mean_independence(x, y, degree=3):
    """
    Compare:
    X -> Y  : residual from Y~X should satisfy E[e|X] ~ 0
    Y -> X  : residual from X~Y should generally fail more
    """
    score_xy = mean_independence_score(x, y, degree=degree)
    score_yx = mean_independence_score(y, x, degree=degree)
    return "X->Y" if score_xy < score_yx else "Y->X"


# 3. Standard LiNGAM
def direction_lingam(x, y):
    data = np.column_stack([x, y])
    model = DirectLiNGAM()
    model.fit(data)

    # causal_order gives indices of causes first
    # index 0 = X, index 1 = Y
    order = model.causal_order_
    return "X->Y" if order == [0, 1] else "Y->X"


# 4. One simulation experiment
def run_simulation(n=500, beta=1.0, scenario="gaussian", reps=500, degree=3, seed=123):
    rng = np.random.default_rng(seed)

    results = []

    for _ in range(reps):
        sim_seed = rng.integers(0, 10**9)
        x, y = simulate_xy(n=n, beta=beta, scenario=scenario, seed=sim_seed)

        pred_lingam = direction_lingam(x, y)
        pred_meanind = direction_mean_independence(x, y, degree=degree)

        results.append({
            "scenario": scenario,
            "n": n,
            "beta": beta,
            "lingam_correct": int(pred_lingam == "X->Y"),
            "meanind_correct": int(pred_meanind == "X->Y")
        })

    return pd.DataFrame(results)



# 5. Run across settings
all_results = []

for scenario in ["gaussian", "non_gaussian"]:
    for n in [100, 300, 1000]:
        df_res = run_simulation(
            n=n,
            beta=1.0,
            scenario=scenario,
            reps=300,      
            degree=3,
            seed=42 + n
        )
        all_results.append(df_res)

results = pd.concat(all_results, ignore_index=True)

# 6. Summary table
summary = (
    results
    .groupby(["scenario", "n"], as_index=False)
    .agg(
        lingam_success_rate=("lingam_correct", "mean"),
        meanind_success_rate=("meanind_correct", "mean")
    )
)

print("\nSuccess rates (proportion of correct direction X->Y):\n")
print(summary)

# 7. Simple plo
fig, ax = plt.subplots(figsize=(8, 5))

for scenario in summary["scenario"].unique():
    sub = summary[summary["scenario"] == scenario]
    ax.plot(sub["n"], sub["lingam_success_rate"], marker="o", label=f"LiNGAM - {scenario}")
    ax.plot(sub["n"], sub["meanind_success_rate"], marker="s", linestyle="--", label=f"Mean-ind - {scenario}")

ax.set_xlabel("Sample size")
ax.set_ylabel("Success rate")
ax.set_title("Causal direction recovery: X -> Y")
ax.set_ylim(0, 1.05)
ax.legend()
plt.show()